In [1]:
print("\n📚 Installing dependencies...")
!pip install -q torch torchvision torchaudio
!pip install -q numpy matplotlib seaborn
!pip install -q pygame pymunk
!pip install -q opencv-python
print("✓ Dependencies installed")


📚 Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00a 0:00:01
✓ Dependencies installed


In [ ]:
# ========================================
# COMPLETE GOOGLE COLAB TRAINING CELL
# CS780 OBELIX Competition - D3QN-PER Agent
# ========================================

import os
import subprocess
import sys
from pathlib import Path
from datetime import datetime

# Step 1: Clone repository
print("📦 Cloning repository...")
if not os.path.exists("CS780-OBELIX"):
    !git clone https://github.com/arpity22/CS780-OBELIX.git
    os.chdir("CS780-OBELIX")
else:
    os.chdir("CS780-OBELIX")
    !git pull
print("✓ Repository ready")

# Step 2: Install dependencies

# Step 3: Verify files exist
print("\n🔍 Checking required files...")
required_files = ["obelix.py", "train_d3qn_per.py", "agent_d3qn_per.py"]
for file in required_files:
    if Path(file).exists():
        print(f"  ✓ {file}")
    else:
        print(f"  ❌ {file} NOT FOUND!")
        sys.exit(1)

# Step 4: Check GPU availability
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n💻 Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   Running on CPU (slower but works fine)")

# Step 5: Start training
print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)

# Training configuration
training_config = {
    "agent_id": "colab_level1",
    "episodes": 3000,
    "difficulty": 0,
    "num_envs": 4,  # Use 4 parallel envs on Colab (CPU has limited cores)
    "device": device,
    "multi_seed": True,
    "save_freq": 100,
}

SUB_DIR = f"submission_{training_config['agent_id']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# Build command
cmd = [
    "python", "train_d3qn_per.py",
    "--obelix_py", "./obelix.py",
    "--agent_id", training_config["agent_id"],
    "--episodes", str(training_config["episodes"]),
    "--difficulty", str(training_config["difficulty"]),
    "--num_envs", str(training_config["num_envs"]),
    "--device", training_config["device"],
    "--save_freq", str(training_config["save_freq"]),
]

# Add multi-seed if enabled
if training_config["multi_seed"]:
    cmd.extend(["--multi_seed", "--seed_list", "42", "123", "456", "789", "999"])

# Add GPU-specific flags
if device == "cuda":
    cmd.append("--use_amp")  # Enable mixed precision on GPU

print("\n📋 Training Configuration:")
for key, value in training_config.items():
    print(f"   {key}: {value}")

print(f"\n⏱️  Expected time: ~2-3 hours")
print(f"📊 Check submission_{training_config['agent_id']}/ folder for results")
print("\n" + "="*60 + "\n")

# Execute training
try:
    subprocess.run(cmd, check=True)
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETE!")
    print("="*60)
    
    # Show results
    if os.path.exists(SUB_DIR):
        print(f"\n📂 Output files in: {SUB_DIR}/")
        files = os.listdir(SUB_DIR)
        for f in sorted(files):
            print(f"   - {f}")
        
        # Show training log summary
        log_file = Path(SUB_DIR) / "training_log.txt"
        if log_file.exists():
            print("\n📊 Training Summary:")
            with open(log_file, 'r') as f:
                lines = f.readlines()
                # Print last 20 lines (summary)
                print("".join(lines[-20:]))
    
except subprocess.CalledProcessError as e:
    print(f"\n❌ Training failed with error code {e.returncode}")
    print("Check the error messages above")
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")
finally:
    # Cleanup
    if device == "cuda":
        torch.cuda.empty_cache()

print("\n" + "="*60)
print("Next steps:")
print("1. Check diagnostic plots in submission folder")
print("2. Download weights.pth and agent_d3qn_per.py")
print("3. Submit to Codabench competition")
print("="*60)

In [ ]:
# ========================================
# DOWNLOAD RESULTS
# ========================================

from google.colab import files
import shutil

# Create zip of submission folder
submission_dir = SUB_DIR  # Change to your agent_id
zip_name = "obelix_agent_level1"

if os.path.exists(submission_dir):
    print(f"📦 Creating zip file: {zip_name}.zip")
    shutil.make_archive(zip_name, 'zip', submission_dir)
    
    print(f"⬇️  Downloading {zip_name}.zip...")
    files.download(f"{zip_name}.zip")
    print("✅ Download complete!")
    
    print(f"\n📋 Files included:")
    for f in os.listdir(submission_dir):
        print(f"   - {f}")
else:
    print(f"❌ Directory {submission_dir} not found!")